# VQCC Hedge Wizardry

**What this is.** An ongoing public collection of *small* verified tweaks to circuits for the [verifiable quantum circuit compilation competition](https://www.kaggle.com/competitions/google-verifiable-quantum-circuit-compilation). A tweak lands here when it passes the official verifier but its expected gain is below +0.1% of the overall score. Gains that small are too slight to hold back, so they are released straight away: the circuit in the [VQCC Hedge Wizardry dataset](https://www.kaggle.com/datasets/manwithacat/vqcc-hedge-wizardry), the explanation here. Entries will be added over time, one section each. The version history of this notebook and the dataset is the dated record.

Each entry is:

- **explained**: what changed and why it pays, written for someone new to the task;
- **verified**: the unchanged official verifier passes the circuit, on Kaggle, in the cells at the end. For DQI the gain is also judged on its expected cost over 48 fixed random draws, never on one lucky draw;
- **credited**: most tweaks are made on someone else's public circuit, and that author is named first, with the licence kept.

**It is also a working submission notebook.** The hidden cells at the end take the cheapest public circuit for each task from our [VQCC public circuits dataset](https://www.kaggle.com/datasets/manwithacat/vqcc-apache-public-circuits). Where an entry is cheaper for its task, they swap it in, check every file's SHA-256, run the unchanged official verifiers and write `submission.zip` with a receipt.

**How to read it.** The code is hidden; click a cell to see it. Numbers carry the evidence tier they were measured at, and a number is never shown at a higher tier than that:

- *static model*: arithmetic on the official cost formula;
- *local engine*: our fast re-implementation of the simulator, cross-checked against the official verifier;
- *local official verifier*: the unmodified competition verifier on a local build of kickmix;
- *official verifier (Kaggle)*: the same verifier run by this notebook on Kaggle.

Bigger changes, and the method most entries reuse, are explained in [Quantum Voodoo](https://www.kaggle.com/code/manwithacat/vqcc-quantum-voodoo).

In [ ]:
def rank_cost(r):
    # Cost used to rank a manifest row within its task (lower is better); None if unscored. A DQI cost at one salt is
    # one draw of its sampled failures (one or two phase errors move R from 1.19126 to 1.19673 / 1.20139), so DQI
    # ranks by dqi_expected_cost (mean over a 48-draw sweep) when known, else dqi_zero_failure_cost (R floor x Q^2 x
    # mean Toffolis), else local_task_score (older manifests). invsqrt and chemistry rank by local_task_score.
    keys = ("dqi_expected_cost", "dqi_zero_failure_cost", "local_task_score") if r.get("task") == "dqi" \
        else ("local_task_score",)
    for k in keys:
        try:
            v = float(r.get(k) or "nan")
        except ValueError:
            continue
        if v == v:
            return v
    return None

def rank_basis(r):
    # Which column rank_cost used, for the tables
    if r.get("task") != "dqi":
        return "default salt"
    return ("expected (48 draws)" if r.get("dqi_expected_cost") else
            "zero-failure" if r.get("dqi_zero_failure_cost") else "default salt")

# Tie rule: rows within REL_TIE (relative) of the task's cheapest ranking cost are a tie, and the tie goes to the
# earliest public circuit (the original). A byte-different copy of a circuit changes the verifier's measurement
# draw, so it can come out a hair cheaper by noise alone; it should not displace the original's designer.
import datetime as _dt
REL_TIE = 1e-5

def public_when(r):
    # First non-empty, readable public_since / first_known / release_date as UTC (date only = 00:00 UTC); else None
    for k in ("public_since", "first_known", "release_date"):
        s = (r.get(k) or "").strip()
        if not s:
            continue
        try:
            d = _dt.datetime.fromisoformat(s.replace("Z", "+00:00"))
        except ValueError:
            continue
        return d.replace(tzinfo=_dt.timezone.utc) if d.tzinfo is None else d.astimezone(_dt.timezone.utc)
    return None

def rank_order(rows):
    # Scored rows of one task: the tie group (cost <= cheapest x (1 + REL_TIE)) earliest public first (missing dates
    # last, then lowest cost, then sha256), then the rest by cost, then sha256
    scored = [(rank_cost(r), r) for r in rows]
    scored = [(c, r) for c, r in scored if c is not None]
    if not scored:
        return []
    limit = min(c for c, _ in scored) * (1 + REL_TIE)
    far = _dt.datetime.max.replace(tzinfo=_dt.timezone.utc)
    def tie_key(cr):
        w = public_when(cr[1])
        return (w is None, w or far, cr[0], cr[1].get("sha256") or cr[1].get("sha12") or "")
    tied = sorted((cr for cr in scored if cr[0] <= limit), key=tie_key)
    rest = sorted((cr for cr in scored if cr[0] > limit),
                  key=lambda cr: (cr[0], cr[1].get("sha256") or cr[1].get("sha12") or ""))
    return [r for _, r in tied + rest]

def pick_best(rows_for_task):
    # The row to use for a task: rank_order()[0], or None if no row is scored
    order = rank_order(rows_for_task)
    return order[0] if order else None


import base64, csv, glob, os, hashlib, json, shutil, sys, time, zipfile
from math import prod
from pathlib import Path

T0 = time.monotonic()
import numpy as np
import pandas as pd
from IPython.display import HTML, display

TASK_NAMES = {"invsqrt": "Inverse square root", "dqi": "DQI decoder", "chemistry": "Chemistry (dfthcsos)"}

def show(df, height=None):
    # A scrollable HTML table: readable on the notebook page, with live links
    html = df.to_html(index=False, render_links=True, escape=True, float_format=lambda x: f"{x:,.6g}")
    style = f"max-height:{height}px;overflow:auto;" if height else "overflow:auto;"
    display(HTML(f'<div style="{style}">{html}</div>'))

INPUT = Path(os.environ.get("VQCC_NB_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("VQCC_NB_WORK", "/kaggle/working"))
SUB = Path(os.environ.get("VQCC_NB_SUB", "/tmp/submission"))   # assembled here; only submission.zip and check.json go to Output
WHEEL_GLOB = "kickmix-*-manylinux*x86_64*.whl"
# Organisers' reference costs (the score is the geometric mean of reference / ours over the three tasks)
REFERENCE = {"invsqrt": 2076774.233715282, "dqi": 24439463583672.93, "chemistry": 3355599588.3155375}
VERIFIER_TASK = {"invsqrt": "invsqrt", "dqi": "dqi", "chemistry": "dfthcsos"}

def find(pattern, root=INPUT):
    hits = sorted(Path(p) for p in glob.glob(f"{root}/**/{pattern}", recursive=True))  # follows symlinked mounts
    if not hits:
        raise FileNotFoundError(f"{pattern} not found under {root}")
    return hits

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(1 << 20):
            h.update(block)
    return h.hexdigest()

def family_sha256(directory):
    # The dataset's chemistry family hash: SHA-256 of sorted '<relative path>\t<file sha256>\n' lines
    files = sorted(p for p in Path(directory).rglob("*") if p.is_file())
    text = "".join(f"{p.relative_to(directory).as_posix()}\t{sha256(p)}\n" for p in files)
    return hashlib.sha256(text.encode()).hexdigest()

In [ ]:
# Figures live in the Hedge Wizardry dataset's figures/ folder.
def figure(name, caption=""):
    hits = sorted(Path(p) for p in glob.glob(f"{INPUT}/**/figures/{name}", recursive=True))
    if not hits:
        print(f"[figure {name} is not in the attached dataset version]")
        return
    mime = "image/gif" if name.endswith(".gif") else "image/png"
    data = base64.b64encode(hits[0].read_bytes()).decode()
    cap = f'<figcaption style="color:#52514e;font-size:13px;margin-top:4px">{caption}</figcaption>' if caption else ""
    display(HTML(f'<figure style="margin:10px 0"><img src="data:{mime};base64,{data}" '
                 f'style="max-width:100%;height:auto" alt="{name}"/>{cap}</figure>'))

## 2. The entries

One row per tweak, read from the dataset's `entries.csv`, which also holds the full verifier and draw results and every gate, met or missed. ΔQ is the change in qubits and ΔT the change in mean Toffolis per shot. Expected GM is the change in the expected overall score, with the public frontier's circuits for the other two tasks.

In [ ]:
hw_csv = find("entries.csv")[0]
HW_DIR = hw_csv.parent
HW = list(csv.DictReader(open(hw_csv, newline="")))
show(pd.DataFrame([{
    "entry": e["id"], "date": e["date"], "task": e["task"], "change": e["title"],
    "base (author, circuit)": e["base_circuit"], "new circuit": e["candidate_sha12"],
    "ΔQ": int(e["delta_qubits"]), "ΔT per shot": f"{float(e['delta_toffoli_mean']):+,.1f}",
    "expected GM": f"{float(e['expected_gm_gain_pct']):+.3f}%",
    "failing draws (base → new)": (f"{e['draws_failing_base']} → {e['draws_failing_candidate']} of 48"
                                   if e["draws_failing_base"] else "no random draw"),
    "gates": ("missed: " + e["gates_missed"].split(":")[0]) if e["gates_missed"] else "all met",
    "base notebook": e["base_notebook"]} for e in HW]))
print(f"{len(HW)} entr{'y' if len(HW) == 1 else 'ies'} in {hw_csv.name} (sha256 {sha256(hw_csv)[:16]}...)")

## 3. Entry HW-01: yoyobar's DQI decoder, 889 → 888 qubits

**In one line:** on yoyobar's public 889-qubit DQI circuit `73e1b187b288`, our release-and-recompute frees one qubit for +460.4 Toffolis per shot, against a break-even of about 960. The result, `5faba4591853`, has 888 qubits and costs 0.117% less at the default draw. Expected gain over 48 draws: about +0.036% on the overall score. One preregistered gate was missed, and this section says which one and why.

### The task, briefly

The DQI task asks for a reversible *decoder*. The circuit reads 70 syndrome words of 12 bits each (840 input qubits). It finds up to 35 errors among 4,095 positions of a Reed–Solomon code over the field GF(2¹²), applies a phase that depends on the answer, and then has to restore every input exactly. The verifier runs 102,400 random shots. The cost is

$$\text{cost} = R \times Q^2 \times T$$

Here Q is the number of qubits, T the mean number of Toffoli gates per shot, and R a penalty factor for failed shots. The circuits are long lists of operations on bits: in each shot every wire holds 0 or 1, plus a running sign.

### Why one qubit is worth about 960 Toffolis here

Because Q is squared, taking one qubit off 889 changes the cost by a factor (888/889)², a cut of about 0.225%. To get the same cut from Toffolis alone you would have to remove 0.225% of T, and with T ≈ 427,500 per shot that is about 2T/Q ≈ **960 Toffolis** (our experiment card rounded it to 965). So any change that frees a qubit for fewer than about 960 extra Toffolis per shot lowers the cost. (Static model: arithmetic on the cost formula.)

### Release-and-recompute, in plain terms

A circuit's width Q is its *peak*: the most wires alive at any one moment. A wire that holds a value nobody will look at for a long time still counts towards that peak.

- **Release.** Such a wire can be cleaned early by measuring it sideways (an *HMR*, measurement in the X basis). The wire goes back to 0 and the simulator records a random bit. If the value was 1 and the bit came out 1, the sign of that shot flips: a *phase debt*. A conditional Z repays the debt, and Z is a Clifford gate, so it is free. Releasing therefore costs nothing at the time.
- **Recompute.** The value is now gone, and the random bit says nothing about it. When the circuit needs it again, it must compute it again, and that costs Toffolis.

The trade pays when the wire was idle across *every* widest stretch of the circuit, so that the peak really falls, and when the recompute costs less than the qubit is worth. We developed this line on our own decoder (889 → 888, explained in section 3 of [Quantum Voodoo](https://www.kaggle.com/code/manwithacat/vqcc-quantum-voodoo)). HW-01 ports it to yoyobar's circuit.

### What changed in yoyobar's circuit

yoyobar's decoder (on SomeJoint's line) works through 8 "epochs". Each epoch has a one-bit *pole* flag that records whether a 24-bit denominator was zero. A census of the circuit found 984 stretches where it is 889 wires wide. In 907 of them, one wire sits idle: **q852, the epoch-0 pole**. It is computed early, in the forward pass (extract), as [D = 0] by a 23-Toffoli zero test. It is then held for about 45 million operations until the reverse pass (unextract) needs it. The other 77 stretches are the 8 reverse-pass epochs, the same pattern our earlier census edits had already solved on our own decoder.

The edits, all ours, on yoyobar's anchors:

- **q852 is released in extract and recomputed later** as q866 ∧ [N = 0], a zero test on a 24-bit register N combined with the bit q866. We checked the formula on 3,072 sampled shots (seeds 101/102/103), and it holds on every one just before the point where it is needed. It fails at every other cut we tested, so the recompute sits exactly there. Then a conditional Z repays the release's phase debt.
- **The earlier census edits are ported to the 8 reverse-pass epochs:** each pole is released after its first use and recomputed as [D = 0]; a 12-wire "norm" is released and recomputed as Sq(a) + a·b + x¹¹·Sq(b); a small GF(8) inverse is done in place.
- **q878 gets one unconditional reset** after a conditional block, so the allocator can reuse the wire.
- **Each pole-handover chain gets an early erase.** yoyobar builds the next epoch's pole (old pole ∧ [N = 0], a 24-step chain) before erasing the old one. A first build was still 889 wide at 21 operations, the 7 chain tops. Erasing the chain's first link as soon as the second exists costs 0 Toffolis. The second link's erasure then repays a CCZ under its bit, which costs 0.5 T on average.

A certified lifetime allocator re-packed the wires and proved the result equivalent, with a lower bound of 888. The measured price is **+460.4 mean Toffolis per shot** (same draw, local engine), for one qubit worth about 960. The projection (888/889)² × (1 + 460.4 / 427,475) predicts −0.1174%, and −0.1174% is what was measured.

### The ledger's view

Our circuit viewer, `render.py`, has a *ledger* mode (in the kickmix-macos fork): a liveness analysis that lists each value's lifetime and how long it sits idle while the circuit is at its peak width, which is what the cost charges. A value released by measurement-based uncomputation (an HMR here) leaves a phase correction to apply later, which the ledger tracks as an open record. **Before anything was built**, the ledger listed q852 as the top idle hold in yoyobar's circuit: open in extract and idle at the circuit's peak width for most of the run (907 of its 984 full-width stretches).

The figure is the ledger's comparison of base and candidate. At this size one qubit is worth about 994 Toffoli-equivalents. The recompute costs +545 Toffoli-equivalents of static Toffolis (the static count, which counts every gate once, so it differs from the 460 measured per shot). There are +280 new HMR records, repaid by +232 fix-ups. Net −449 Toffoli-equivalents, a cost ratio of 0.99898. All the extra work sits in the reverse stages (unsynthesis, unextract). Tier: static model.

**Why we stopped at one qubit.** The ledger's next leads were q866 (about 752 Toffoli-equivalents) and a block of 7 wires idle from extract to unsynthesis (about 655 Toffoli-equivalents each). A census of the 888-qubit candidate, though, finds 34 stretches at width 888 where *every* live wire is in use. They are 24-bit multiplications that hold both inputs and the product. No release can lower that peak, and a release that does not lower Q only adds Toffolis. So we stopped.

In [ ]:
figure("hw01_ledger_73e1b187_vs_5faba459.png", "render.py ledger --compare 73e1b187b288 5faba4591853 (kickmix-macos, "
       "static model). Capital = width × the qubit's worth; expense = static Toffolis; per-stage deltas below.")

### Results

| | Base `73e1b187` (yoyobar) | HW-01 `5faba459` | Tier |
| --- | ---: | ---: | --- |
| Qubits | 889 | **888** | local official verifier |
| Mean Toffolis per shot, default draw | 427,474.9 | 427,935.6 | local official verifier |
| Cost at the default (empty) salt | 402,458,290,262 | **401,986,129,571** (−0.117%), PASS, 0 errors | local official verifier |
| Failing draws out of 48 fixed salts | 7 | **8** | local engine |
| Mean cost over the 48 draws | 402,727,802,153 | **402,293,640,703** (−0.108%; cheaper on 47 of 48) | local engine |
| Overall score with the public frontier's invsqrt and chemistry, default salt | 23.17173 | **23.18080** (+0.039%) | arithmetic on local verifier costs |
| Expected overall score over the 48 draws | 23.16658 | **23.17491** (+0.036%) | arithmetic on local engine draws |

*Local official verifier* means the unchanged competition verifier on our local build of kickmix, which has matched the hosted one digit for digit. The cells at the end of this notebook re-run it on Kaggle. *Local engine* is our faster re-implementation of the simulator, cross-checked against the official verifier. None of these numbers is a leaderboard score.

**The gate we missed.** Before building, the experiment card fixed its gates. One was "failing draws ≤ 7", the base's count, and HW-01 has 8, so **that gate is missed**. Every value error is identical draw for draw. The extra failure is a single phase error in draw `sweep-42` (batch 341, shot 6). We replayed that batch under 64 different measurement seeds: the shot fails in 32 of 64 runs for the base and in 33 of 64 for HW-01. It is a coin-flip input for *both* circuits. The base happened to pass it on this seed, and HW-01's extra measurements shift the random stream, so it landed on the other side. Our reading (local engine replay) is that the expected failure rate is unchanged, which makes the same-draw −0.117% the better estimate of the gain. The gate still counts as missed, and we publish with that stated. We did not re-roll the file or make a byte variant to get a luckier draw.

Every other gate passed: the allocator (equivalent, 888), the engine and official cross-check, the official verifier, the file contract (header, registers, the same failure set on a fixed-weight sweep), and a 48-draw mean cost below the base's.

The per-draw numbers are in the dataset under `evidence/HW-01/`.

### Credits for HW-01

- **The base circuit is yoyobar's**: `73e1b187b288`, published in [LB 22.99 | invsqrt 101 - 62 qubits](https://www.kaggle.com/code/yoyobar/lb-22-99-invsqrt-101-62-qubits) (version 8) under Apache-2.0, on the decoder line of **SomeJoint (Agniv Banerjee)** ([v19](https://www.kaggle.com/code/somejoint/quantum-circuits-22-75-local-gm-897-qubit-dqi)). The idea of releasing a pole flag and recomputing it when needed was first used publicly by **heliosli** on SomeJoint's decoder ([v11](https://www.kaggle.com/code/heliosli/the-dataset-is-public-now-enjoy)); it is credited in Quantum Voodoo.
- **The tweak is ours** (manwithacat): the release-and-recompute method of [Quantum Voodoo](https://www.kaggle.com/code/manwithacat/vqcc-quantum-voodoo) (our 7f3d64c1 → 40fa4ebf), re-anchored on yoyobar's circuit, plus the q852 release, the q878 reset and the handover erase. We describe it as "our tweak to yoyobar's circuit", not as our circuit.
- The work was done with substantial help from Claude (Anthropic): it wrote most of the tooling and analysis under my direction, and I chose the questions, approved each experiment and checked the claims.

## 3b. Entry HW-03: chemistry weights, a constrained refit alternated with the column gauge

**In one line:** the circuits stay as they are: the 15 lookup circuits and the angle-word table of the public chemistry family `ec11a966fb64` are byte-identical. Only the weights in `submission.npz` change. λ_eff falls from 48.293182 to **48.274616**, and the chemistry cost by **-0.0384%**. That is about +0.013% on the overall score, so it is a Hedge Wizardry entry, well below the +0.1% bar.

### The task, briefly

The chemistry task (`dfthcsos`) asks for a factorized Hamiltonian of the FeMoCo molecule plus circuits that load its angles. The cost is

$$\text{cost} = \lambda_{\text{eff}} \times Q \times T$$

Here Q is the qubit count and T the Toffoli count of the full model. Both depend only on the circuits, which are unchanged here (Q = 430, T = 16,266). So the cost falls exactly in proportion to λ_eff, which the verifier computes from the weights as

$$\lambda_{\text{eff}} = \sqrt{E_{\text{gap}}\,(2\Lambda - E_{\text{gap}})}$$

Λ is a one-norm: the eigenvalue sum of a one-body matrix plus a two-body term. With 8 ranks r and 60 columns c, the two-body term is a quarter of the sum, over every (r, c), of the *square of a column's L1 norm*. The L1 norm is taken down that column's 143 weights plus one offset. Two reconstruction errors must stay at or below 20 mHa, and E_gap must stay above 3.914898.

### Two moves

- **The column gauge (Isabe11aM00re's idea).** For each rank, the verifier's two error terms and E_gap use the 60 columns only through sums that do not change if you rotate the columns by any orthogonal 60 × 60 matrix. Λ does change, because an L1 norm is not rotation invariant. So a rotation can lower Λ for free. [Isabe11aM00re](https://www.kaggle.com/code/isabe11am00re/qcc-v47-submission) found such a rotation for Ahmed Berat Özer's weights: that is the public `ec11a966fb64` family, and it lowered λ_eff by 0.0025. We confirmed it exactly: a Procrustes fit leaves a residual of 1e-14.
- **The constrained refit (ours).** A projected L-BFGS on log λ_eff over all weights. It holds e1, e2 and E_gap as equality constraints just inside the limits (19.9999 mHa, and floor + 1e-6), with a Gauss–Newton step back onto those constraints after every move. The words and the circuits never change.

The two moves help each other. The refit moves the weights within the current column frame, and the gauge then finds a sparser frame for the next refit. We alternated them, starting from `ec11a966fb64`:

| step | stage | λ_eff | note |
| ---: | --- | ---: | --- |
| 0 | start | 48.293182 | public base ec11a966fb64 |
| 1 | refit 1 | 48.280459 | 4,000 evaluations, budget |
| 2 | gauge 1 | 48.279101 | exact Jacobi + smoothed-L1 L-BFGS |
| 3 | refit 2 | 48.275613 | 4,000 evaluations, budget |
| 4 | gauge 2 | 48.275192 |  |
| 5 | refit 3 | 48.274750 | stalled after 1,362 evaluations |
| 6 | gauge 3 | 48.274645 |  |
| 7 | refit 4 | 48.274645 | stalled after 119 evaluations, no gain |
| 8 | gauge 4 (final) | 48.274616 | HW-03 candidate |

A control run (refit only, no gauge, same budget) ended at 48.276699, against 48.275613 with the gauge in between. So the gauge adds a little each round on top of the refit (local prescreen, the official evaluator's functions).

**How far can a gauge go?** Since Λ's two-body term only ever loses to a rotation, a rotation-invariant lower bound is the column norms' Frobenius sum. That bound is about 8.6× below today's value, far too loose to be a useful ceiling, so it neither rules the gauge out nor in. Empirically, a multi-start search from Özer's weights (exact Jacobi sweeps, plus smoothed-L1 L-BFGS from the identity, Isabe11aM00re's frame and 128 random kicks) found nothing better than Isabe11aM00re's frame. Most of HW-03's gain comes from the refit.

### Results

| | Base `ec11a966fb64` (Isabe11aM00re / Özer) | HW-03 `e148cacdd4ad` | Tier |
| --- | ---: | ---: | --- |
| error_1 (mHa, limit 20) | 19.999754 | 19.999900 | local official verifier |
| error_2 (mHa, limit 20) | 19.999817 | 19.999900 | local official verifier |
| E_gap − 3.914898 | 1.029e-06 | 1.000e-06 | local official verifier |
| Λ | 299.823549 | 299.594573 | local official verifier |
| λ_eff | 48.293182 | **48.274616** | local official verifier |
| Qubits, Toffolis | 430, 16,266 | 430, 16,266 | local official verifier |
| Chemistry cost | 337,780,864.96 | **337,651,009.06** (-0.0384%), PASS | local official verifier |
| Overall score with the public frontier's invsqrt and DQI, default salt | 23.18120 | **23.18417** | arithmetic on local verifier costs |

The margins are thinner than the base's: e1 and e2 sit 1e-4 mHa inside the limit instead of about 2e-4. They are deterministic, though: the chemistry verifier decodes the same integer words every time, and its errors do not depend on a random draw. The cells at the end re-run the verifier on Kaggle.

### Credits for HW-03

- **Weights: Ahmed Berat Özer** (`4eabe756bc36`, [Quantum AI - DF-THC Group Compression](https://www.kaggle.com/code/ahmedberatozer/quantum-ai-df-thc-group-compression), Apache-2.0).
- **The per-rank column-gauge idea: Isabe11aM00re** (`ec11a966fb64`, [QCC v47 submission](https://www.kaggle.com/code/isabe11am00re/qcc-v47-submission) v3, Apache-2.0). HW-03 starts from that point.
- **Word table: Atakan Aldemir** (table `4e3c8f079137`). **The (R, B, C) = (8, 143, 60) factorization: SomeJoint (Agniv Banerjee).** **Loaders: Heliosli, Dmitrii Gluzdov and Atakan Aldemir.**
- **Ours** (manwithacat): the constrained refit, alternating it with the gauge, and the gauge search. We describe HW-03 as our tweak to their weights, not as our circuit.
- The work was done with substantial help from Claude (Anthropic): it wrote most of the tooling and analysis under my direction, and I chose the questions, approved each experiment and checked the claims.

## 4. What did not work

Ideas that failed are kept with the same care as those that worked, in the [VQCC Negative Results](https://www.kaggle.com/datasets/manwithacat/vqcc-chemistry-negative-results) dataset: one row per idea, with its evidence tier, why it failed and what would reopen it. Before trying something, check there.

HW-01 added two rows to our ledger (NR-0119 and NR-0120), and HW-03 one (NR-0131, which corrects our earlier NR-0080). They appear in the dataset from its next version.

- **Porting our failure fix to yoyobar's circuit gains nothing.** Our Quantum Voodoo failure fix rebuilt decoder groups 4–7 as homogeneous chains with their poles dropped. yoyobar's circuit already has groups 3–7 built that way, so there is no Toffoli saving left and no chance-zero failure to remove (measured per group, local engine).
- **Releasing more idle wires cannot reach 887.** Releasing q866 or the 7-wire block cannot lower the peak, because 34 stretches at 888 use every live wire (24-bit multiplications; census, structural).
- **A column gauge alone does not beat Isabe11aM00re's (HW-03).** From Özer's weights, exact Jacobi sweeps and a smoothed-L1 L-BFGS from the identity, from Isabe11aM00re's frame and from 128 random kicks found nothing below Isabe11aM00re's frame, and Haar-random starts end far above it (local prescreen).
- **Our earlier claim that the fixed-word refit sits at a constrained optimum (NR-0080) was start-specific.** From Heliosli's weights it stalled; from Özer's and Isabe11aM00re's it keeps improving (HW-03).

## 5. How to reproduce

1. **Get the circuits.** Attach the [VQCC Hedge Wizardry dataset](https://www.kaggle.com/datasets/manwithacat/vqcc-hedge-wizardry). `entries.csv` gives each circuit's SHA-256 (`SHA256SUMS` lists every file). The base circuits are in the [VQCC public circuits dataset](https://www.kaggle.com/datasets/manwithacat/vqcc-apache-public-circuits) under `circuits/<task>/<sha12>.kmb`.
2. **Verify on Kaggle.** Copy this notebook and run it (CPU, internet off). It installs our [kickmix fork](https://github.com/manwithacat/kickmix-macos) offline from the attached wheel and runs the unchanged official verifiers on the portfolio. The entry gate below checks each entry's cost against `entries.csv`. The verifier seeds every shot from the file's SHA-256, so the same bytes give the same cost at the default salt.
3. **Verify locally.** With kickmix built from our fork (macOS) or the official Linux build, run the official verifier on `circuits/dqi/5faba4591853.kmb` with an empty salt: it should report PASS, 888 qubits and cost 401,986,129,571.42. For HW-03, run the official `dfthcsos` verifier on the folder `circuits/chemistry/<sha12>/` (15 loaders and `submission.npz`); its cost is listed in `entries.csv`.
4. **The 48 draws** come from our local engine with fixed seeds shared by base and candidate. That engine is not published, so these numbers cannot be re-run from public files. The per-draw costs and errors are in `evidence/<entry>/draws_48.csv`.
5. **Read the change.** A `.kmb` circuit is a list of operations. The fork's circuit viewer (`tools/circuit-viewer/render.py`, including `ledger --compare`) shows where base and candidate differ.

## The submission this notebook makes

The hidden cells below:

1. pick the cheapest public circuit per task from the public dataset's manifest (chemistry: a verified assembly of public parts if that is cheaper than the best single family);
2. for each task, swap in the cheapest Hedge Wizardry entry if it is cheaper than the public pick, after checking its SHA-256;
3. install our kickmix fork offline;
4. run the unmodified official verifiers at full settings;
5. verify every entry's circuit and match its cost to `entries.csv` (the entry gate);
6. write `submission.zip` only if everything passes, with `check.json`, `entry_gates.json` and `receipt.json`.

In [ ]:
manifest = find("manifest.csv")[0]
DATA = manifest.parent
rows = list(csv.DictReader(open(manifest, newline="")))
print(f"manifest: {manifest} ({len(rows)} circuits, sha256 {sha256(manifest)[:16]}...)")

picks = {}
for task in REFERENCE:
    scored = rank_order([r for r in rows if r["task"] == task])  # tie group first, original first
    picks[task] = scored[0]
    best_cost = min(rank_cost(r) for r in scored)
    ties = [r["sha12"] for r in scored[1:] if rank_cost(r) <= best_cost * (1 + REL_TIE)]
    print(f"{task:9} {len(scored):2} candidates; picked {scored[0]['sha12']} "
          f"ranking cost {rank_cost(scored[0]):.6e} ({rank_basis(scored[0])}); "
          f"default-salt cost {float(scored[0]['local_task_score']):.6e}"
          + (f"; ties within noise, earlier public kept: {', '.join(ties)}" if ties else ""))

# Chemistry: a verified assembly of public parts replaces the best single family only if it is cheaper
ASSEMBLY = None
best_json = DATA / "parts" / "assembly_best.json"
if best_json.is_file():
    a = json.loads(best_json.read_text())
    single = float(picks["chemistry"]["local_task_score"])
    print(f"parts/assembly_best.json: verified {a['verified_cost']:.6e} ({a['verify_date']}) "
          f"vs best single family {picks['chemistry']['sha12']} {single:.6e}")
    if a.get("verified_status") == "PASS" and float(a["verified_cost"]) < single:
        ASSEMBLY = a

shutil.rmtree(SUB, ignore_errors=True)
SUB.mkdir(parents=True)
for task, r in picks.items():
    if task == "chemistry" and ASSEMBLY:
        fam = SUB / "dfthcsos"
        fam.mkdir()
        files = [(ASSEMBLY["npz"]["dataset_path"], ASSEMBLY["npz"]["sha256"], "submission.npz")]
        files += [(s["dataset_path"], s["file_sha256"], f"angle_k_{s['slot']}.kmb") for s in ASSEMBLY["slots"]]
        for path, digest, name in files:
            assert sha256(DATA / path) == digest, f"{path} does not match parts/assembly_best.json"
            shutil.copyfile(DATA / path, fam / name)
        authors = sorted({s["author"] for s in ASSEMBLY["slots"]})
        picks["chemistry"] = {
            "task": "chemistry", "sha12": "assembled", "sha256": family_sha256(fam),
            "qubits": ASSEMBLY["qubits"], "local_task_score": str(ASSEMBLY["verified_cost"]),
            "author": "assembled: loaders " + ", ".join(authors) + "; weights " + ASSEMBLY["npz"]["author"],
            "notebook_title": "assembly of public chemistry parts (parts/assembly_best.json)",
            "source_notebook_url": "https://www.kaggle.com/datasets/manwithacat/vqcc-apache-public-circuits",
            "source_notebook_version": "", "release_date": ASSEMBLY["verify_date"],
            "core_author": ASSEMBLY["word_table_origin"]["author"],
            "change_summary": f"cheapest public loader per slot with the best weights on word table "
                              f"{ASSEMBLY['table_key']}", "also_found_in": ""}
        print(f"chemistry: assembled from {len(files)} dataset files, every SHA-256 checked")
    elif task == "chemistry":
        src = DATA / "circuits" / "chemistry" / r["sha12"]
        assert family_sha256(src) == r["sha256"], f"chemistry family {r['sha12']} does not match the manifest"
        shutil.copytree(src, SUB / "dfthcsos")
    else:
        src = DATA / "circuits" / task / f"{r['sha12']}.kmb"
        assert sha256(src) == r["sha256"], f"{task} {r['sha12']} does not match the manifest"
        shutil.copyfile(src, SUB / f"{task}.kmb")
print("assembled and hash-checked:", sorted(p.name for p in SUB.iterdir()))

pd.set_option("display.max_colwidth", 80)
display(pd.DataFrame([{"task": t, "candidates": sum(1 for x in rows if x["task"] == t and x["local_task_score"]),
                       "picked": r["sha12"], "qubits": r.get("qubits"),
                       "ranking cost": rank_cost(r), "ranked by": rank_basis(r),
                       "local cost (default salt)": float(r["local_task_score"]), "author": r["author"],
                       "sha256 checked": True} for t, r in picks.items()]).set_index("task"))

In [ ]:
# Hedge Wizardry entries: verify each file's SHA-256, then let the cheapest entry per task replace the public
# pick only if it beats it on the same ranking cost (DQI: the 48-draw mean when the entry has one, against the
# public pick's rank_cost; others: the default-salt cost) by more than the tie threshold REL_TIE.
HW_FILES = {}
for e in HW:
    src = HW_DIR / e["dataset_path"]
    got = family_sha256(src) if e["task"] == "chemistry" else sha256(src)
    assert got == e["candidate_sha256"], f"{e['id']}: {src.name} does not match entries.csv"
    HW_FILES[e["id"]] = src
HW_USED = {}
for task in REFERENCE:
    def entry_cost(e):
        return float(e.get("draws_mean_cost_candidate") or e["local_task_score"]) if task == "dqi"                         else float(e["local_task_score"])
    mine = sorted((e for e in HW if e["task"] == task), key=entry_cost)
    if not mine:
        continue
    e, public = mine[0], picks[task]
    pub_cost = rank_cost(public)
    if not entry_cost(e) < pub_cost * (1 - REL_TIE):
        print(f"{task}: public {public['sha12']} ({pub_cost:.6e}, {rank_basis(public)}) stays; best entry "
              f"{e['id']} {e['candidate_sha12']} ({entry_cost(e):.6e}) does not beat it beyond the tie threshold")
        continue
    src = HW_FILES[e["id"]]
    if task == "chemistry":
        shutil.rmtree(SUB / "dfthcsos", ignore_errors=True)
        shutil.copytree(src, SUB / "dfthcsos")
        ASSEMBLY = None
    else:
        shutil.copyfile(src, SUB / f"{task}.kmb")
    picks[task] = {
        "task": task, "sha12": e["candidate_sha12"], "sha256": e["candidate_sha256"], "qubits": e["qubits"],
        "local_task_score": e["local_task_score"],
        "author": f"manwithacat (Hedge Wizardry {e['id']}) on {e['base_author']}'s {e['base_sha256'][:12]}",
        "notebook_title": f"VQCC Hedge Wizardry {e['id']}: {e['title']}",
        "source_notebook_url": "https://www.kaggle.com/code/manwithacat/vqcc-hedge-wizardry-notebook",
        "source_notebook_version": "", "release_date": e["date"], "core_author": e["base_author"],
        "change_summary": e["card_summary"], "also_found_in": ""}
    HW_USED[task] = e["id"]
    print(f"{task}: Hedge Wizardry {e['id']} {e['candidate_sha12']} ({e['qubits']} qubits, local cost "
          f"{float(e['local_task_score']):.6e}) replaces public {public['sha12']} by {public['author']} "
          f"({pub_cost:.6e}, {rank_basis(public)}): {entry_cost(e) / pub_cost - 1:+.3%}")

In [ ]:
tool_dirs = [Path(p).parent for root in ("/kaggle/usr/lib", "/kaggle/input")
             for p in sorted(glob.glob(f"{root}/**/kickmix_tools.py", recursive=True))]
assert tool_dirs, "attach the kickmix_tools utility script"
sys.path.insert(0, str(tool_dirs[0]))
import kickmix_tools as kt
print("kickmix_tools", kt.__version__, "from", tool_dirs[0])

wheel_dir = find(WHEEL_GLOB)[0].parent
FORK_DIR = kt.install(str(wheel_dir), target="/tmp/kickmix_fork")
FORK_PROVENANCE = kt.provenance()
print("fork provenance:", FORK_PROVENANCE)
print("competition data:", kt.competition_dir())

In [ ]:
PATHS = {"invsqrt": SUB / "invsqrt.kmb", "dqi": SUB / "dqi.kmb", "chemistry": SUB / "dfthcsos"}
results, check_rows = {}, []
for task, circuit in PATHS.items():
    r = kt.verify(VERIFIER_TASK[task], circuit, build="fork")
    results[task] = r
    res = r["result"]
    listed = float(picks[task]["local_task_score"])
    check_rows.append({
        "task": task, "circuit": picks[task]["sha12"], "status": res["status"],
        "cost": res["score"], "manifest cost": listed, "cost / manifest": res["score"] / listed,
        "times cheaper than reference": REFERENCE[task] / res["score"],
        "qubits": res.get("num_qubits"), "Toffolis (mean)": res.get("toffoli_count"),
        "value errors": res.get("value_errors") if task == "dqi" else None,
        "phase errors": res.get("phase_errors") if task == "dqi" else None,
        "wall s": round(r["wall_seconds"], 1),
        "peak MB": round(r["peak_pss_sum_mb"] or r["peak_rss_sum_mb"] or 0)})
checks = pd.DataFrame(check_rows).set_index("task")
FORMATS = {"cost": "{:.6e}", "manifest cost": "{:.6e}", "cost / manifest": "{:.9f}",
           "times cheaper than reference": "{:.4f}", "qubits": "{:.0f}", "Toffolis (mean)": "{:,.1f}",
           "value errors": "{:.0f}", "phase errors": "{:.0f}", "wall s": "{:.1f}", "peak MB": "{:,.0f}"}
shown = checks.copy()
for col, fmt in FORMATS.items():  # value/phase errors exist for DQI only; blank cells show as —
    shown[col] = [fmt.format(v) if pd.notna(v) else "—" for v in checks[col]]
display(shown)
print("— = not applicable: only the DQI verifier counts value and phase errors.")
failed = [t for t, r in results.items() if r["result"]["status"] != "PASS"]
assert not failed, f"not writing a submission: {failed} did not pass"

ratios = {t: REFERENCE[t] / results[t]["result"]["score"] for t in REFERENCE}
gm = prod(ratios.values()) ** (1 / 3)
# DQI cost depends on the draw. The headline uses the DQI ranking cost (mean over a 48-draw sweep, or the
# zero-failure cost); gm is this run's single default-salt draw (2026-10-31: the grader uses its own hidden draw).
ratios_exp = dict(ratios, dqi=REFERENCE["dqi"] / rank_cost(picks["dqi"]))
gm_exp = prod(ratios_exp.values()) ** (1 / 3)
zf = picks['dqi'].get('dqi_zero_failure_cost')  # best case: no failed DQI shots in a draw
gm_zero = prod(dict(ratios, dqi=REFERENCE['dqi'] / float(zf)).values()) ** (1 / 3) if zf else None
print(f"expected geometric mean: {gm_exp:.5f} (DQI {rank_basis(picks['dqi'])}); "
      f"this run's single default-salt draw: {gm:.5f}")
checks.to_csv(WORK / "checks.csv")

### Entry gate

Every entry's circuit, verified here by the unchanged official verifier. An entry already in the portfolio reuses the run above, and any other is verified separately. Its cost must match the local record in `entries.csv` to 1e-9, which holds because the verifier seeds every shot from the file's SHA-256.

In [ ]:
ENTRY_GATES = []
for e in HW:
    task = e["task"]
    if picks[task]["sha256"] == e["candidate_sha256"]:
        res, how = results[task]["result"], "portfolio run"
    else:
        res, how = kt.verify(VERIFIER_TASK[task], HW_FILES[e["id"]], build="fork")["result"], "separate run"
    listed = float(e["local_task_score"])
    rel = abs(res["score"] / listed - 1)
    ENTRY_GATES.append({
        "entry": e["id"], "circuit": e["candidate_sha12"], "verifier run": how, "status": res["status"],
        "qubits": res.get("num_qubits"), "Toffolis (mean)": f"{res.get('toffoli_count', float('nan')):,.1f}",
        "cost (Kaggle)": f"{res['score']:.6e}", "entries.csv cost": f"{listed:.6e}",
        "matches to 1e-9": rel <= 1e-9, "in this submission": HW_USED.get(task) == e["id"]})
show(pd.DataFrame(ENTRY_GATES))
assert all(g["status"] == "PASS" for g in ENTRY_GATES), "an entry failed the official verifier"
(WORK / "entry_gates.json").write_text(json.dumps(ENTRY_GATES, indent=1, default=str))

In [ ]:
members = [("invsqrt.kmb", SUB / "invsqrt.kmb"), ("dqi.kmb", SUB / "dqi.kmb")]
members += [(f"dfthcsos/{p.relative_to(SUB / 'dfthcsos').as_posix()}", p)
            for p in sorted((SUB / "dfthcsos").rglob("*")) if p.is_file()]
with zipfile.ZipFile(WORK / "submission.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for name, path in members:
        z.write(path, name)
(WORK / "check.json").write_text(json.dumps({
    "manifest_sha256": sha256(manifest), "submission_sha256": sha256(WORK / "submission.zip"),
    "geometric_mean_local": gm, "fork": FORK_PROVENANCE,
    "picks": {t: {k: picks[t][k] for k in ("sha256", "author", "notebook_title", "source_notebook_url",
                                            "release_date", "local_task_score")} for t in picks},
    "chemistry_assembly": ASSEMBLY,
    "results": {t: r["result"] for t, r in results.items()}}, indent=1, default=str))
shutil.rmtree(SUB)
print("wrote", WORK / "submission.zip", "and", WORK / "check.json")

## Receipt

One table that says what was submitted and why it can be trusted. Each gate is a check this run made, not a claim copied from elsewhere: the file is the one the manifest lists (by SHA-256; for an assembled chemistry entry, every file is the one `parts/assembly_best.json` lists), the unchanged official verifier passed it, and its cost matches the dataset's record of the same verifier on the same bytes. Saved as `receipt.json`.

In [ ]:
WALL = time.monotonic() - T0
receipt = pd.DataFrame([{
    "task": TASK_NAMES[t], "circuit": picks[t]["sha12"], "sha256": picks[t]["sha256"][:16] + "…",
    "published by": picks[t]["author"], "version": picks[t]["source_notebook_version"],
    "verdict": results[t]["result"]["status"],
    "times cheaper than reference": REFERENCE[t] / results[t]["result"]["score"],
    "link": picks[t]["source_notebook_url"]} for t in REFERENCE])
show(receipt)
CHEM_PARTS = None
if ASSEMBLY:
    print("Chemistry was assembled from public parts: each slot's loader, the weights and the word table's origin")
    CHEM_PARTS = pd.DataFrame(
        [{"part": f"slot {s['slot']} loader", "file": s["dataset_path"], "sha256": s["file_sha256"][:16] + "…",
          "published by": s["author"], "version": s.get("notebook_version", ""),
          "Toffolis per shot": s.get("toffoli_per_shot_source"), "link": s["notebook"]} for s in ASSEMBLY["slots"]]
        + [{"part": "weights (submission.npz)", "file": ASSEMBLY["npz"]["dataset_path"],
            "sha256": ASSEMBLY["npz"]["sha256"][:16] + "…", "published by": ASSEMBLY["npz"]["author"],
            "version": ASSEMBLY["npz"].get("notebook_version", ""), "Toffolis per shot": None,
            "link": ASSEMBLY["npz"]["notebook"]},
           {"part": f"word table {ASSEMBLY['table_key']} (origin)", "file": "parts/word_tables.csv",
            "sha256": "", "published by": ASSEMBLY["word_table_origin"]["author"], "version": "",
            "Toffolis per shot": None, "link": ASSEMBLY["word_table_origin"]["notebook"]}])
    show(CHEM_PARTS)
rel = {t: abs(results[t]["result"]["score"] / float(picks[t]["local_task_score"]) - 1) for t in REFERENCE}
GATES = {
    "G1 every file matches its listed SHA-256": True,  # asserted when the files were assembled
    "G2 the official verifier passed all three": all(r["result"]["status"] == "PASS" for r in results.values()),
    "G3 every cost within 1e-9 of the manifest record": max(rel.values()) <= 1e-9,
    "G4 notebook wall time under 2 hours": WALL <= 7200,
}
show(pd.DataFrame([{"gate": g, "result": "pass" if ok else "FAIL"} for g, ok in GATES.items()]))
print(f"this run's default-salt geometric mean {gm:.5f}; expected {gm_exp:.5f}; wall time {WALL / 60:.1f} min. "
      f"G3 means each cost was reproduced exactly at the default salt (the local verifier seeds every shot from "
      f"the file's SHA-256).")
print("Not a leaderboard prediction: the grader draws DQI shots with its own hidden salt (fixed per circuit, "
      "different from the local default), so its DQI failure count can differ from this run's. The expected "
      "geometric mean above is the fair estimate.")

# Headroom: how close each pick is to the verifier's pass/fail limits (from this run's verifier output)
E1_MAX = E2_MAX = 0.020          # Ha, both chemistry accuracy gates
GAP_FLOOR = 3.914898             # strict E_gap floor in the dfthcsos verifier
head = []
inv = results["invsqrt"]["result"]
w = inv.get("worst_error_over_tolerance")
if w is not None:
    head.append({"task": TASK_NAMES["invsqrt"], "check": "worst error / tolerance (must stay below 1)",
                 "value": f"{w:.5f}", "margin": f"{(1 - w) * 100:.3f}% of tolerance",
                 "tight": (1 - w) < 0.005})
dq = results["dqi"]["result"]
head.append({"task": TASK_NAMES["dqi"], "check": "failed shots on this file's draw (value / phase)",
             "value": f"{dq.get('value_errors')} / {dq.get('phase_errors')}",
             "margin": f"repetition factor R = {dq.get('repetition_factor', float('nan')):.5f}; "
                       "each extra failure costs about 0.46% of DQI cost",
             "tight": (dq.get("value_errors") or 0) + (dq.get("phase_errors") or 0) > 0})
ch = results["chemistry"]["result"].get("metrics", {})
for key, lim in (("error_1", E1_MAX), ("error_2", E2_MAX)):
    if key in ch:
        m = (lim - ch[key]) * 1000
        head.append({"task": TASK_NAMES["chemistry"], "check": f"{key} (must stay ≤ 20 mHa)",
                     "value": f"{ch[key] * 1000:.5f} mHa", "margin": f"{m:.5f} mHa", "tight": m < 0.01})
if "E_gap" in ch:
    m = ch["E_gap"] - GAP_FLOOR
    head.append({"task": TASK_NAMES["chemistry"], "check": "E_gap (must stay above 3.914898)",
                 "value": f"{ch['E_gap']:.7f}", "margin": f"{m:.2e}", "tight": m < 1e-4})
hd = pd.DataFrame(head)
hd["status"] = ["tight" if t else "comfortable" for t in hd.pop("tight")]
print("Headroom: how close each pick is to failing the verifier. 'tight' means a tiny change in the file, the "
      "verifier or floating point could flip it.")
show(hd)

# DQI final-salt risk for the picked file, from the DQI Salt Check notebook's output when it covers this file
SALT = None
for f in sorted(glob.glob(f"{INPUT}/**/dqi_salt_check_summary_*.csv", recursive=True)):
    for r in csv.DictReader(open(f, newline="")):
        if r.get("circuit") == picks["dqi"]["sha12"]:
            SALT = dict(r, file=Path(f).name)
if SALT:
    print(f"DQI final-salt risk for {picks['dqi']['sha12']} ({SALT['runs']} salts in {SALT['file']}): "
          f"errors per salt {SALT['errors per salt']}; P(0 failures) {SALT['P(0 err)']}; expected GM factor "
          f"{SALT['E[GM factor]']} (95% {SALT['GM factor 95%']}).")
else:
    print(f"DQI final-salt risk for {picks['dqi']['sha12']}: not measured yet (this file is not in the attached "
          f"DQI Salt Check output).")
if not GATES["G3 every cost within 1e-9 of the manifest record"]:
    print("G3 failed: this run's verifier disagrees with the manifest, so treat the local score with care.")
(WORK / "receipt.json").write_text(json.dumps({
    "gates": GATES, "geometric_mean_local": gm, "headroom": head, "dqi_salt_check": SALT, "wall_seconds": WALL, "relative_cost_difference": rel,
    "submission_sha256": sha256(WORK / "submission.zip"), "manifest_sha256": sha256(manifest),
    "rows": receipt.to_dict(orient="records"),
    "chemistry_parts": CHEM_PARTS.to_dict(orient="records") if CHEM_PARTS is not None else None},
    indent=1, default=str))